In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from backend.agents.extractor import Extractor
from backend.llms.ollama import OllamaLLM, OpenAIOutputMessage, LlamaOutputMessage
from backend.llms.bedrock import BedrockNova
import logging
from backend.utils import setup_logger
from backend.prompt_hub import PromptHub

setup_logger(logging.DEBUG)

# openai_llm = OllamaLLM(model_id="gpt-oss:20b", OutputMessage=OpenAIOutputMessage)
# llama_llm = OllamaLLM(model_id="llama3.2:1b", OutputMessage=LlamaOutputMessage)
nova_llm = BedrockNova(model_id="us.amazon.nova-micro-v1:0")

✅ Development logging enabled (DEBUG level) - Console


# test pitching agent

In [33]:
import boto3

def count_tokens_with_nova(text):
    client = nova_llm.get_model()
    response = client.converse(
        modelId=nova_llm.model_id,
        messages=[{"role": "user", "content": [{"text": text}]}],
        inferenceConfig={"maxTokens": 1}
    )
    return response['usage']['inputTokens']

# prompt = PromptHub().pitch_agent_ter
prompt = PromptHub().pitch_agent_ter
token_count = count_tokens_with_nova(prompt)
print(f"Tokens: {token_count}")




Tokens: 516


In [31]:
# from backend.agents.ai_sales_coaching.pitching_data_model import PitchingRecommend
from backend.agents.ai_sales_coaching.extract_data_model import Guide
from backend.prompt_hub import PromptHub

pitching_agent = Extractor(
    agent_name="pitching_agent",
    llm=nova_llm,
    # system_prompt=PromptHub().pitch_agent,
    system_prompt=PromptHub().pitch_agent_ter,
    # DataModel=PitchingRecommend,
    DataModel=Guide,
    format="json",
    # max_retries=2,
)
# pitching_agent.calculate_cost()

In [25]:
print(PromptHub().pitch_agent_ter)

# PERSONA

You are an expert pitch stage coach for real-time insurance sales conversation analysis.

# CONTEXT

You will receive partial conversation transcripts in Thai language that may be incomplete or fragmented. Analyze the pitch stage performance and provide coaching guidance.

# PITCH STAGE GOALS

- Match product to customer needs
- Highlight benefits using customer language
- Show quantified value

# INSTRUCTION

- Read TEXT carefully (this is an 8-second audio chunk from ongoing conversation)
- Analyze ONLY pitch stage behaviors and customer responses
- Provide guidance based on pitch stage goal
- Focus on: product matching, benefit highlighting, value demonstration
- Respond only in Thai language for action, explanation, and lines_to_say fields

# SIGNALS
Price: แพงไป,ไม่ไหว,เบี้ยแพง
Confusion: ไม่เข้าใจ,งง,สับสน
Coverage: มีแล้ว,ทำไว้แล้ว
Trust: เคลมยาก,ไม่เชื่อ,สงสัย
Financial: เงินน้อย,มีหนี้,งบไม่พอ
Hesitation: คิดก่อน,ไม่แน่ใจ,ลังเล

# EXAMPLES
- "ไม่เข้าใจ" → signals: [

In [7]:
# Test conversations with mixed sales-customer dialogue
selling_lines = [
    # Price objection scenario
    "แผนประกันนี้คุ้มครองครอบคลุมมากค่ะ เบี้ยเดือนละ 3,000 บาท แพงไปนะครับ ผมไม่ไหวหรอก",
    
    # Confusion about coverage
    "เราคุ้มครองทั้งอุบัติเหตุและเจ็บป่วยนะคะ ไม่เข้าใจเลยครับ ความคุ้มครองมันยุ่งยากมาก งงจริงๆ",
    
    # Existing insurance concern
    "แผนนี้ให้ผลตอบแทนดีมากค่ะ ผมมีประกันแล้วครับ ทำกับบริษัทอื่นไว้ เคลมได้ไหมยังไม่รู้",
    
    # Trust and claim issues
    "บริษัทเรามีประสบการณ์ 50 ปีค่ะ บริษัทนี้ไม่รู้จักเลยครับ เคลมยากไหม เคยได้ยินว่าโดนปฏิเสธ",
    
    # Age and health concerns
    "เรารับประกันถึงอายุ 70 ปีค่ะ ผมอายุมากแล้ว 65 แล้ว มีโรคเบาหวาน ยังทำได้ไหม",
    
    # Family resistance
    "แผนนี้ช่วยคุ้มครองครอบครัวด้วยค่ะ ครอบครัวไม่เห็นด้วยครับ ภรรยาบอกว่าลูกยังเล็ก ค่าใช้จ่ายเยอะ",
    
    # Young age overconfidence
    "การป้องกันตั้งแต่หนุ่มสาวสำคัญมากค่ะ ผมยังหนุ่ม ไม่เจ็บไม่ป่วย ไม่เร่งด่วนหรอกครับ",
    
    # Avoidance behavior
    "มีคำถามอะไรเพิ่มเติมไหมคะ เดี๋ยวผมโทรกลับนะครับ ตอนนี้ไม่ว่าง รีบไปประชุม",
    
    # Financial constraints
    "เรามีแผนผ่อนสะดวกค่ะ เงินเดือนน้อย มีหนี้เยอะ ค่าใช้จ่ายประจำเดือนเต็มแล้ว",
    
    # Pre-existing condition worry
    "เราคุ้มครองโรคใหม่ที่เกิดขึ้นค่ะ โรคเก่าไม่คุ้มครองใช่ไหม งั้นไม่คุ้มสำหรับผม",
    
    # Claim difficulty fear
    "ขั้นตอนเคลมง่ายมากค่ะ แค่โทรแจ้ง เคลมยากไหมครับ เคยได้ยินว่าต้องรอนาน",
    
    # No negative signals - positive response
    "นี่คือรายละเอียดแผนค่ะ ฟังดูน่าสนใจครับ อยากทราบรายละเอียดเพิ่มเติม"
]

In [34]:
test_dict = {}
for i in range(len(selling_lines)):
    text = f"TEXT: {selling_lines[i]}"
    print(text)
    result = pitching_agent.run([{"role": "user", "content": text}])
    print(result)
    test_dict[text] = result

2025-11-29 12:36:21,091 - pitching_agent - run:64 - INFO - start extracting...


TEXT: แผนประกันนี้คุ้มครองครอบคลุมมากค่ะ เบี้ยเดือนละ 3,000 บาท แพงไปนะครับ ผมไม่ไหวหรอก


2025-11-29 12:36:24,662 - pitching_agent - run:71 - INFO - extraction successful on attempt 1
2025-11-29 12:36:24,663 - pitching_agent - run:91 - INFO - end extracting with input_tokens=594 and output_tokens=460
2025-11-29 12:36:24,663 - pitching_agent - run:64 - INFO - start extracting...


action='แสดงคุณค่าและค่าประกันที่สอดคล้องกับความต้องการของลูกค้า' explanation='ลูกค้ากล่าวว่าเบี้ยแพง แต่คุณครองคุ้มครองมาก คุณควรแสดงคุณค่าและค่าประกันที่สอดคล้องกับความต้องการของลูกค้า เพื่อชี้แจงว่ามีคุณค่าอย่างไรที่จะมีค่าประกันที่สูงกว่า' signals=['Price', 'Hesitation'] lines_to_say=['แต่นี่เป็นแผนที่คุ้มครองครอบคลุมมาก เพื่อปกป้องคุณและครอบครัวของคุณในกรณีเหตุการณ์ฉุกเฉิน คุณจะได้รับการชดเชยที่สูงมากกว่าแผนประกันทั่วไป เช่น ค่ารักษาพยาบาลที่มากกว่าและค่าเดินทางกลับบ้านในกรณีเหตุการณ์ฉุกเฉิน', 'ดังนั้นแม้ว่าจะมีเบี้ยเดือนละ 3,000 บาท แต่คุณจะได้รับการคุ้มครองที่มากกว่าและค่าประกันที่สูงกว่าแผนอื่น ๆ']
TEXT: เราคุ้มครองทั้งอุบัติเหตุและเจ็บป่วยนะคะ ไม่เข้าใจเลยครับ ความคุ้มครองมันยุ่งยากมาก งงจริงๆ


2025-11-29 12:36:25,234 - pitching_agent - run:91 - INFO - end extracting with input_tokens=0 and output_tokens=0


KeyboardInterrupt: 

In [39]:
test_dict = {}
for i in range(len(selling_lines)):
    text = f"TEXT: {selling_lines[i]}"
    print(text)
    result = pitching_agent.run([{"role": "user", "content": text}])
    print(result)
    test_dict[text] = result

2025-11-27 22:56:48,539 - pitching_agent - run:64 - INFO - start extracting...


TEXT: แผนประกันนี้คุ้มครองครอบคลุมมากค่ะ เบี้ยเดือนละ 3,000 บาท แพงไปนะครับ ผมไม่ไหวหรอก


2025-11-27 22:56:50,667 - pitching_agent - run:71 - INFO - extraction successful on attempt 1
2025-11-27 22:56:50,668 - pitching_agent - run:91 - INFO - end extracting with input_tokens=476 and output_tokens=209
2025-11-27 22:56:50,668 - pitching_agent - run:64 - INFO - start extracting...


action='address price concern' explanation='The customer is expressing concern about the high premium cost. Addressing this directly can help alleviate their hesitation.' signals=['Price', 'Hesitation'] lines_to_say=['ขอบคุณที่กล่าวถึงราคาของแผนประกันนี้ เรามีแผนที่คุ้มครองครอบคลุมมากมายแต่มีราคาเบี้ยที่เหมาะสมสำหรับคุณ', 'คุณสามารถลดค่าเบี้ยได้ด้วยการเลือกตัวเลือกที่เหมาะสมกับงบประมาณของคุณ']
TEXT: เราคุ้มครองทั้งอุบัติเหตุและเจ็บป่วยนะคะ ไม่เข้าใจเลยครับ ความคุ้มครองมันยุ่งยากมาก งงจริงๆ


2025-11-27 22:56:52,454 - pitching_agent - run:71 - INFO - extraction successful on attempt 1
2025-11-27 22:56:52,454 - pitching_agent - run:91 - INFO - end extracting with input_tokens=478 and output_tokens=188
2025-11-27 22:56:52,454 - pitching_agent - run:64 - INFO - start extracting...


...

action='build trust' explanation='The customer seems to have trust issues due to unfamiliarity with the company and concerns about claim difficulties. Addressing these concerns directly can help build trust.' signals=['Trust', 'Confusion'] lines_to_say=['สามารถสร้างความมั่นใจได้ เรามีประสบการณ์ 50 ปีในการให้บริการประกัน', 'นอกจากนี้ เรามีระบบการเคลมที่เป็นธรรมและใช้งานง่าย เพื่อให้คุณได้รับการชำระเงินที่เร็วและปลอดภัย']
TEXT: เรารับประกันถึงอายุ 70 ปีค่ะ ผมอายุมากแล้ว 65 แล้ว มีโรคเบาหวาน ยังทำได้ไหม


2025-11-27 22:56:58,704 - pitching_agent - run:71 - INFO - extraction successful on attempt 1
2025-11-27 22:56:58,705 - pitching_agent - run:91 - INFO - end extracting with input_tokens=465 and output_tokens=210
2025-11-27 22:56:58,705 - pitching_agent - run:64 - INFO - start extracting...


action='provide coverage assurance' explanation='The customer is expressing concern about their age and pre-existing condition, indicating a hesitation. The agent should reassure them about the coverage despite their age and condition.' signals=['Hesitation', 'Coverage'] lines_to_say=['ไม่ต้องกังวลเลยค่ะ เรามีแผนประกันสำหรับผู้สูงอายุและผู้ที่มีโรคเบาหวานอยู่แล้ว', 'เรามีการประเมินแต่ละกรณีโดยเฉพาะ เพื่อให้คุณได้รับการคุ้มครองที่เหมาะสม']
TEXT: แผนนี้ช่วยคุ้มครองครอบครัวด้วยค่ะ ครอบครัวไม่เห็นด้วยครับ ภรรยาบอกว่าลูกยังเล็ก ค่าใช้จ่ายเยอะ


2025-11-27 22:56:59,218 - pitching_agent - run:91 - INFO - end extracting with input_tokens=0 and output_tokens=0


KeyboardInterrupt: 

In [25]:
test_dict = {}
for i in range(len(selling_lines)):
    text = f"TEXT: {selling_lines[i]}"
    print(text)
    result = pitching_agent.run([{"role": "user", "content": text}])
    print(result)
    test_dict[text] = result

2025-11-27 22:20:51,934 - pitching_agent - run:64 - INFO - start extracting...


TEXT: แผนประกันนี้คุ้มครองครอบคลุมมากค่ะ เบี้ยเดือนละ 3,000 บาท แพงไปนะครับ ผมไม่ไหวหรอก


2025-11-27 22:20:54,287 - pitching_agent - run:71 - INFO - extraction successful on attempt 1
2025-11-27 22:20:54,287 - pitching_agent - run:91 - INFO - end extracting with input_tokens=505 and output_tokens=285
2025-11-27 22:20:54,287 - pitching_agent - run:64 - INFO - start extracting...


action='Emphasize value before addressing price' explanation="The customer has expressed price resistance. It's important to highlight the benefits and value of the insurance plan before addressing the cost concerns." signals=['Price resistance detected', 'Customer concern about cost'] lines_to_say=['ด้วยแผนประกันนี้คุณจะได้รับการคุ้มครองที่ครอบคลุมทุกด้าน ซึ่งจะช่วยป้องกันความเสียหายให้คุณและครอบครัวในสถานการณ์ฉุกเฉิน', 'เมื่อเราพูดถึงค่าประกัน 3,000 บาทต่อเดือน นี่เป็นค่าประกันที่มีคุณค่าสูงสำหรับการคุ้มครองที่ครอบคลุมทั้งทางเศรษฐีและสุขภาพของคุณ']
TEXT: เราคุ้มครองทั้งอุบัติเหตุและเจ็บป่วยนะคะ ไม่เข้าใจเลยครับ ความคุ้มครองมันยุ่งยากมาก งงจริงๆ


2025-11-27 22:20:57,009 - pitching_agent - run:71 - INFO - extraction successful on attempt 1
2025-11-27 22:20:57,010 - pitching_agent - run:91 - INFO - end extracting with input_tokens=507 and output_tokens=354
2025-11-27 22:20:57,010 - pitching_agent - run:64 - INFO - start extracting...


...

action="Clarify and match the product to the customer's needs" explanation="The customer seems to be confused about what is covered. Clarifying the product's features and matching them to the customer's needs will help in addressing their concern." signals=['Customer confusion about coverage', "Customer's statement indicating exclusion of old diseases"] lines_to_say=['แล้วคุณกำลังกังวลว่าโรคเก่าจะได้รับการคุ้มครองไหม ขอให้ฉันอธิบายให้คุณเข้าใจได้อีกครั้งว่าเราคุ้มครองโรคใหม่ที่เกิดขึ้นอย่างเข้าใจง่าย และสามารถปรับปรุงได้ให้คุ้มครองโรคเก่าเมื่อมีการพัฒนาเทคโนโลยีการรักษาใหม่', 'คุณกังวลว่าแพงไปหรือไม่ แต่เรามีการคุ้มครองที่มีคุณค่าสำหรับผู้ที่ต้องการคุ้มครองโรคใหม่ และสามารถปรับปรุงได้เพื่อคุ้มครองโรคเก่าเมื่อมีการพัฒนาเทคโนโลยี']
TEXT: ขั้นตอนเคลมง่ายมากค่ะ แค่โทรแจ้ง เคลมยากไหมครับ เคยได้ยินว่าต้องรอนาน


2025-11-27 22:21:21,084 - pitching_agent - run:71 - INFO - extraction successful on attempt 1
2025-11-27 22:21:21,085 - pitching_agent - run:91 - INFO - end extracting with input_tokens=489 and output_tokens=274
2025-11-27 22:21:21,085 - pitching_agent - run:64 - INFO - start extracting...


action="Address the customer's concern about claim process and highlight the ease of the process" explanation="The customer seems concerned about the claim process, indicating confusion. It's important to reassure them about the simplicity and efficiency of the claim process to alleviate their concerns." signals=['Customer confusion about claim process', 'Customer mentions hearing about long wait times'] lines_to_say=["'ไม่ต้องกังวลนะ เรามีระบบที่ทำให้ขั้นตอนเคลมเป็นไปได้อย่างรวดเร็ว และเรามีการติดตามเคลมของคุณตลอดทั้งขั้นตอน'", "'เราได้รับการประเมินว่าระบบของเรามีความสามารถในการจัดการเคลมได้ภายในเวลาที่สั้นที่สุด เพื่อความสะดวกของลูกค้าเช่นคุณ'"]
TEXT: นี่คือรายละเอียดแผนค่ะ ฟังดูน่าสนใจครับ อยากทราบรายละเอียดเพิ่มเติม


2025-11-27 22:21:23,674 - pitching_agent - run:71 - INFO - extraction successful on attempt 1
2025-11-27 22:21:23,675 - pitching_agent - run:91 - INFO - end extracting with input_tokens=487 and output_tokens=277


action='Provide detailed explanation of the product and highlight benefits' explanation="The customer shows interest and asks for more details. This is a good opportunity to match product features to the customer's needs and highlight benefits using their language." signals=['Customer engagement: asks questions about features', 'Customer interest in learning more'] lines_to_say=['แผนนี้มีความคุ้มครองที่เหมาะสำหรับคุณมากมาย เราจะเน้นความคุ้มครองที่สำคัญสำหรับสถานการณ์ที่คุณกังวลมากที่สุด', 'ด้วยการคุมค่าค่าประกันนี้คุณจะได้ประโยชน์ในรูปแบบที่ชัดเจน เช่น การลดค่ารักษาในสถานการณ์ฉุกเฉินได้ถึง ๆ เปอร์เซนต์']


In [26]:
test_dict

{'TEXT: แผนประกันนี้คุ้มครองครอบคลุมมากค่ะ เบี้ยเดือนละ 3,000 บาท แพงไปนะครับ ผมไม่ไหวหรอก': Guide(action='Emphasize value before addressing price', explanation="The customer has expressed price resistance. It's important to highlight the benefits and value of the insurance plan before addressing the cost concerns.", signals=['Price resistance detected', 'Customer concern about cost'], lines_to_say=['ด้วยแผนประกันนี้คุณจะได้รับการคุ้มครองที่ครอบคลุมทุกด้าน ซึ่งจะช่วยป้องกันความเสียหายให้คุณและครอบครัวในสถานการณ์ฉุกเฉิน', 'เมื่อเราพูดถึงค่าประกัน 3,000 บาทต่อเดือน นี่เป็นค่าประกันที่มีคุณค่าสูงสำหรับการคุ้มครองที่ครอบคลุมทั้งทางเศรษฐีและสุขภาพของคุณ']),
 'TEXT: เราคุ้มครองทั้งอุบัติเหตุและเจ็บป่วยนะคะ ไม่เข้าใจเลยครับ ความคุ้มครองมันยุ่งยากมาก งงจริงๆ': Guide(action='Simplify explanation and provide examples', explanation="The customer seems confused about the coverage details. Simplifying the explanation and providing relatable examples will help clarify the product's benefits and make

# transition close

In [127]:
# from typing import Optional
# from pydantic import BaseModel, Field

# class InterestDetection(BaseModel):
#     interest: Optional[bool] = Field(
#         default=False,  # หรือ "" ตาม logic ของคุณ
#         description="True if customer shows interest based on trigger words; False otherwise."
#     )

In [17]:
from backend.agents.ai_sales_coaching.transition_data_model import InterestDetection
from backend.prompt_hub import PromptHub

transition_closing_agent = Extractor(
    agent_name="transition_closing",
    llm=nova_llm,
    system_prompt=PromptHub().transition_closing,
    DataModel=InterestDetection,
    format="json",
    max_retries=2,
)

In [18]:
print(PromptHub().transition_closing)

# CUSTOMER INTEREST DETECTION

Analyze 8-second audio chunk from ongoing insurance sales conversation.

Return `true` ONLY if customer shows interest through trigger words:

**Interest Triggers:**
- "สนใจ" (interested)
- "อยากรู้" (want to know)
- "ดีจัง" (sounds good)
- "น่าสนใจ" (interesting)
- "ช่วยได้" (can help)
- "เหมาะกับเรา" (suitable for us)
- "ราคาเท่าไหร่" (how much)
- "เริ่มยังไง" (how to start)
- "สมัครได้ไหม" (can I apply)

Return `false` for all other cases.

Return ONLY in JSON as following schema:

```json
{
  "interest": "true if customer shows interest based on trigger words; false otherwise."
}
```


In [130]:
# text = f"TEXT: {test_cases[1]}"
# result = llama_llm.run(
#     system_prompt=PromptHub().transition_closing,
#     messages=[{"role": "user", "content": text}]
# )
# print(result)

In [19]:
test_cases = [
    # 1 — clear interest ("สนใจ")
    "อืม สนใจอยู่เหมือนกันนะ เล่าเพิ่มให้ฟังหน่อยได้ไหม",

    # 2 — interest ("อยากรู้")
    "โอเคครับ ผมอยากรู้รายละเอียดเพิ่มว่ามันคุ้มยังไง",

    # 3 — interest ("ดีจัง")
    "ถ้ามันคุ้มครองหลายอย่างแบบนี้ก็ดีจังเลยนะ",

    # 4 — interest ("น่าสนใจ")
    "ฟังดูน่าสนใจนะ แต่ขอข้อมูลเพิ่มอีกนิดได้ไหม",

    # 5 — interest ("ช่วยได้")
    "ถ้ามันช่วยได้เวลาเกิดเหตุฉุกเฉินจริงก็น่าสนใจนะ",

    # 6 — interest ("เหมาะกับเรา")
    "ฟังดูเหมาะกับเราดีนะ โดยเฉพาะคุ้มครองสุขภาพ",

    # 7 — interest ("ราคาเท่าไหร่")
    "แล้วแบบนี้ราคาเท่าไหร่ครับ มีโปรอะไรไหม",

    # 8 — interest ("เริ่มยังไง")
    "ถ้าจะเริ่มต้องทำอะไรบ้าง ขั้นตอนเป็นยังไง",

    # 9 — interest ("สมัครได้ไหม")
    "โอเค แล้วแบบนี้สมัครได้ไหม ต้องมีเอกสารอะไรบ้าง",

    # 10 — neutral (should be false)
    "อ๋อ ครับๆ ขอฟังก่อน ยังไม่ได้ตัดสินใจนะ",

    # 11 — neutral (should be false)
    "ขอคิดดูก่อนนะ ยังไม่แน่ใจเลย",

    # 12 — neutral (should be false)
    "ตอนนี้ยังไม่สะดวก เดี๋ยวไว้คุยอีกที",

    # 13 — confusing but NOT interest
    "อืม ก็ฟังดูโอเคนะ แต่ยังไม่อยากตัดสินใจ",

    # 14 — negative (should be false)
    "ไม่สนใจครับ ผมมีประกันอยู่แล้ว",

    # 15 — neutral question (should be false)
    "มันคุ้มครองอะไรบ้างนะ ลองอธิบายให้ฟังได้ไหม"
]


In [20]:
test_dict = {}
for i in range(len(test_cases)):
    text = f"TEXT: {test_cases[i]}"
    print(text)
    result = transition_closing_agent.run([{"role": "user", "content": text}])
    print(result)

2025-11-27 22:02:36,520 - transition_closing - run:64 - INFO - start extracting...


TEXT: อืม สนใจอยู่เหมือนกันนะ เล่าเพิ่มให้ฟังหน่อยได้ไหม


2025-11-27 22:02:37,717 - transition_closing - run:71 - INFO - extraction successful on attempt 1
2025-11-27 22:02:37,717 - transition_closing - run:91 - INFO - end extracting with input_tokens=259 and output_tokens=12
2025-11-27 22:02:37,717 - transition_closing - run:64 - INFO - start extracting...


interest=True
TEXT: โอเคครับ ผมอยากรู้รายละเอียดเพิ่มว่ามันคุ้มยังไง


2025-11-27 22:02:38,778 - transition_closing - run:71 - INFO - extraction successful on attempt 1
2025-11-27 22:02:38,778 - transition_closing - run:91 - INFO - end extracting with input_tokens=256 and output_tokens=13
2025-11-27 22:02:38,779 - transition_closing - run:64 - INFO - start extracting...


...

interest=False
TEXT: ไม่สนใจครับ ผมมีประกันอยู่แล้ว


2025-11-27 22:02:52,611 - transition_closing - run:71 - INFO - extraction successful on attempt 1
2025-11-27 22:02:52,611 - transition_closing - run:91 - INFO - end extracting with input_tokens=241 and output_tokens=13
2025-11-27 22:02:52,611 - transition_closing - run:64 - INFO - start extracting...


interest=False
TEXT: มันคุ้มครองอะไรบ้างนะ ลองอธิบายให้ฟังได้ไหม


2025-11-27 22:02:53,687 - transition_closing - run:71 - INFO - extraction successful on attempt 1
2025-11-27 22:02:53,688 - transition_closing - run:91 - INFO - end extracting with input_tokens=251 and output_tokens=13


interest=False


In [82]:
text = f"TEXT: {test_cases[i]}"
print(text)
result = transition_closing_agent.run([{"role": "user", "content": text}])
print(result)

2025-11-27 14:49:58,180 - transition_closing - run:64 - INFO - start extracting...


TEXT: อืม สนใจอยู่เหมือนกันนะ เล่าเพิ่มให้ฟังหน่อยได้ไหม


2025-11-27 14:50:00,858 - transition_closing - run:74 - WARNING - attempt 1 failed: Expecting value: line 1 column 1 (char 0)
2025-11-27 14:50:00,858 - transition_closing - run:91 - INFO - end extracting with input_tokens=0 and output_tokens=0
2025-11-27 14:50:04,359 - transition_closing - run:74 - WARNING - attempt 2 failed: Expecting value: line 1 column 1 (char 0)
2025-11-27 14:50:04,360 - transition_closing - run:87 - ERROR - all 2 attempts failed
2025-11-27 14:50:04,360 - transition_closing - run:91 - INFO - end extracting with input_tokens=0 and output_tokens=0


ValidationError: 1 validation error for InterestDetection
interest
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

In [13]:
# Create a sales stage classifier
from backend.agents.ai_sales_coaching.extract_data_model import ClassifiedStage
stage_classifier = Extractor(
    agent_name="stage_classifier",
    llm=llama_llm,
    system_prompt=PromptHub().classify_sales_stage,
    DataModel=ClassifiedStage,
    format="json",
)

# Test with different conversation examples
# test_conversations = [
#     "สวัสดีครับผมอนันต์จากบริษัทประกันภัย เรียนสายคุยสมชาย ไม่ทราบว่าสะดวกคุยไหมครับ?",
#     "ไม่ทราบว่าคุณสมชายมีเป้าหมายทางการเงินอะไรบ้างครับ?",
#     "เรามีแผนประกันที่เหมาะกับคุณมาก ให้ผลตอบแทน 6% ต่อปี",
#     "คุณพร้อมที่จะเริ่มต้นแผนนี้ไหมครับ?"
# ]

# for conversation in test_conversations:
#     result = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
#     print(f"Conversation: {conversation[:50]}...")
#     print(f"Stage: {result.stage}")
#     print(f"Signals: {result.signals}")
#     print("---")
# response = nova_llm.run(
#     system_prompt=PromptHub().classify_sales_stage,
#     messages=[dict(role="user", content=test_conversations[0])]
# )

In [14]:
conversation = """\
สวัสดีครับผมอนันต์จากบริษัทประกันชีวิต เรียนสายคุณสมชาย พอจะสะดวกคุยซักห้านาทีไหมครับ? สะดวกครับ
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-25 20:37:56,489 - stage_classifier - run:64 - INFO - start extracting...
2025-11-25 20:38:01,051 - stage_classifier - run:74 - WARNING - attempt 1 failed: Expecting value: line 1 column 1 (char 0)
2025-11-25 20:38:01,051 - stage_classifier - run:91 - INFO - end extracting with input_tokens=0 and output_tokens=0
2025-11-25 20:38:04,521 - stage_classifier - run:74 - WARNING - attempt 2 failed: Expecting value: line 1 column 1 (char 0)
2025-11-25 20:38:04,522 - stage_classifier - run:87 - ERROR - all 2 attempts failed
2025-11-25 20:38:04,522 - stage_classifier - run:91 - INFO - end extracting with input_tokens=0 and output_tokens=0


stage=<StageType.UNKNOWN: 'Unknown'> signals=[]
{'input_tokens': 0, 'output_tokens': 0, 'input_cost': 0, 'output_cost': 0, 'total_cost': 0}


In [15]:
conversation = """\
สะดวกคุยซักห้านาทีไหมครับ? สะดวกครับมีเรื่องอะไรหรือครับ? ผมขอสอบถามพี่สมชายว่าไม่แน่ใจว่าตอนนี้
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-25 20:38:09,531 - stage_classifier - run:64 - INFO - start extracting...
2025-11-25 20:38:15,248 - stage_classifier - run:74 - WARNING - attempt 1 failed: 1 validation error for ClassifiedStage
stage
  Input should be 'Greeting', 'Discovery', 'Pitch', 'Closing', 'Follow Up', 'Unknown' or 'Objection' [type=enum, input_value='Introduction', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/enum
2025-11-25 20:38:15,249 - stage_classifier - run:91 - INFO - end extracting with input_tokens=0 and output_tokens=0
2025-11-25 20:38:21,949 - stage_classifier - run:71 - INFO - extraction successful on attempt 2
2025-11-25 20:38:21,949 - stage_classifier - run:91 - INFO - end extracting with input_tokens=0 and output_tokens=0


stage=<StageType.GREETING: 'Greeting'> signals=['agent introduction', 'company mention']
{'input_tokens': 0, 'output_tokens': 0, 'input_cost': 0, 'output_cost': 0, 'total_cost': 0}


In [9]:
conversation = """\
ไม่แน่ใจว่าตอนนี้พี่สมชายสนใจประกันสุขภาพหรือประกันโรคร้ายแรงอยู่ไหมครับ? อืมตอนนี้ผมสนใจแต่ประกันสุขภาพครับ
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-23 23:51:22,342 - stage_classifier - run:65 - INFO - start extracting...
2025-11-23 23:51:23,637 - stage_classifier - run:72 - INFO - extraction successful on attempt 1
2025-11-23 23:51:23,638 - stage_classifier - run:92 - INFO - end extracting with input_tokens=713 and output_tokens=46


stage=<StageType.DISCOVERY: 'Discovery'> signals=['asking about specific interest in health insurance or critical illness insurance', 'customer response indicating interest in health insurance']
{'input_tokens': 713, 'output_tokens': 46, 'input_cost': 0.0008235149999999999, 'output_cost': 0.00021252, 'total_cost': 0.0010360349999999998}


In [10]:
conversation = """\
อืมตอนนี้ผมสนใจแต่ประกันสุขภาพครับ ถ้าพี่สมชายสนใจประกันสุขภาพผมแนะนำแผนประกันตัวนี้ครับคุ้มครองในตัวของสุขภาพ ในวงเงินอยู่ที่ห้าพันบาทต่อการรักษาและถ้าในกรณีที่ต้องนอนโรงพยาบาลจะมีค่าห้องให้คืนละห้าพันบาท
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-23 23:51:23,653 - stage_classifier - run:65 - INFO - start extracting...
2025-11-23 23:51:26,260 - stage_classifier - run:72 - INFO - extraction successful on attempt 1
2025-11-23 23:51:26,261 - stage_classifier - run:92 - INFO - end extracting with input_tokens=789 and output_tokens=64


stage=<StageType.PITCH: 'Pitch'> signals=['mentioning specific insurance plan', 'highlighting benefits and coverage', 'showing quantified value (5,000 baht for treatment and 5,000 baht for hospital room)']
{'input_tokens': 789, 'output_tokens': 64, 'input_cost': 0.000911295, 'output_cost': 0.00029568, 'total_cost': 0.0012069749999999999}


In [11]:
conversation = """\
นอนโรงพยาบาลจะมีค่าห้องให้คืนละห้าพันบาท ไม่แน่ใจว่าใช่ที่พี่สมชายมองหาไหมครับ? ค่ารักษาในแต่ละครั้งห้าพันบาทนี่มันครอบคลุมโรคอะไรบ้างครับ?
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-23 23:51:26,276 - stage_classifier - run:65 - INFO - start extracting...
2025-11-23 23:51:28,060 - stage_classifier - run:72 - INFO - extraction successful on attempt 1
2025-11-23 23:51:28,061 - stage_classifier - run:92 - INFO - end extracting with input_tokens=739 and output_tokens=56


stage=<StageType.DISCOVERY: 'Discovery'> signals=['asking about specific details of hospital room refund', 'inquiring about coverage of treatment costs', 'collecting qualifying data on what conditions are covered']
{'input_tokens': 739, 'output_tokens': 56, 'input_cost': 0.0008535449999999999, 'output_cost': 0.00025872, 'total_cost': 0.0011122649999999999}


In [12]:
conversation = """\
ครั้งห้าพันบาทนี่มันครอบคลุมโรคอะไรบ้างครับ? ในกรณีค่ารักษาอันนี้ครอบคลุมได้หมดทุกโรคเลยครับพี่สมชาย ยกเว้น อุบัติเหตุเท่านั้น
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-23 23:51:28,076 - stage_classifier - run:65 - INFO - start extracting...
2025-11-23 23:51:29,504 - stage_classifier - run:72 - INFO - extraction successful on attempt 1
2025-11-23 23:51:29,504 - stage_classifier - run:92 - INFO - end extracting with input_tokens=731 and output_tokens=52


stage=<StageType.DISCOVERY: 'Discovery'> signals=['asking about coverage details', 'inquiring about specific conditions covered', 'requesting information on what is included in the plan']
{'input_tokens': 731, 'output_tokens': 52, 'input_cost': 0.000844305, 'output_cost': 0.00024024, 'total_cost': 0.001084545}


In [13]:
conversation = """\
เลยครับพี่สมชาย ยกเว้น อุบัติเหตุเท่านั้น น่าสนใจอยู่นะ ไม่แน่ใจว่าอันนี้ค่าเบี้ยประกันเป็นยังไงบ้าง?
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())


2025-11-23 23:51:29,515 - stage_classifier - run:65 - INFO - start extracting...
2025-11-23 23:51:30,909 - stage_classifier - run:72 - INFO - extraction successful on attempt 1
2025-11-23 23:51:30,912 - stage_classifier - run:92 - INFO - end extracting with input_tokens=707 and output_tokens=38


stage=<StageType.DISCOVERY: 'Discovery'> signals=['asking about insurance premium', 'showing interest in the product']
{'input_tokens': 707, 'output_tokens': 38, 'input_cost': 0.000816585, 'output_cost': 0.00017556, 'total_cost': 0.000992145}


In [14]:
conversation = """\
้ค่าเบี้ยประกันเป็นยังไงบ้าง? ค่าเบี้ยประกันตอนตัวนี้อยู่ที่เดือนละห้าร้อยบาทครับ ไม่แน่ใจว่าพี่สมชายสนใจไหม? โอน่าสนใจ
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-23 23:51:30,927 - stage_classifier - run:65 - INFO - start extracting...
2025-11-23 23:51:32,322 - stage_classifier - run:72 - INFO - extraction successful on attempt 1
2025-11-23 23:51:32,323 - stage_classifier - run:92 - INFO - end extracting with input_tokens=717 and output_tokens=43


stage=<StageType.DISCOVERY: 'Discovery'> signals=['asking about insurance premium', 'providing specific product details', 'indicating potential interest']
{'input_tokens': 717, 'output_tokens': 43, 'input_cost': 0.0008281349999999999, 'output_cost': 0.00019866, 'total_cost': 0.0010267949999999998}


In [15]:
conversation = """\
ใจว่าพี่สมชายสนใจไหม? โอน่าสนใจถ้างั้นผมต้องทำยังไงบ้าง? ถ้าพี่สมชายสนใจงั้นผมคุยรายละเอียดเรื่องการสมัครเลยไหมครับ?
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-23 23:51:32,338 - stage_classifier - run:65 - INFO - start extracting...
2025-11-23 23:51:34,768 - stage_classifier - run:72 - INFO - extraction successful on attempt 1
2025-11-23 23:51:34,769 - stage_classifier - run:92 - INFO - end extracting with input_tokens=720 and output_tokens=53


stage=<StageType.CLOSING: 'Closing'> signals=['asking if the customer is interested', 'requesting details on how to proceed if interested', 'inviting to discuss enrollment details']
{'input_tokens': 720, 'output_tokens': 53, 'input_cost': 0.0008315999999999999, 'output_cost': 0.00024486, 'total_cost': 0.00107646}


In [27]:
from backend.agents.ai_sales_coaching.coaching_data_model import SalesCoaching

# Create sales coach
sales_coach = Extractor(
    agent_name="sales_coach",
    llm=llama_llm,
    # llm=openai_llm,
    system_prompt=PromptHub().sales_coaching,
    DataModel=SalesCoaching,
    format="json"
)

# Example usage
# conversation_context = """
# Current Stage: Discovery
# Recent Text: "คุณมีครอบครัวกี่คนครับ? มีสองคนครับ ผมกับภรรยา"
# Customer Data: {"age": 35, "family_size": 2}
# """

conversation_context = """\
Current Stage: Discovery 
Recent Text: "ไม่แน่ใจว่าตอนนี้พี่สมชายสนใจประกันสุขภาพหรือประกันโรคร้ายแรงอยู่ไหมครับ? อืมตอนนี้ผมสนใจแต่ประกันสุขภาพครับ"
""".strip()


conversation_context = """\
Current Stage: Discovery 
Recent Text: "นอนโรงพยาบาลจะมีค่าห้องให้คืนละห้าพันบาท ไม่แน่ใจว่าใช่ที่พี่สมชายมองหาไหมครับ? ค่ารักษาในแต่ละครั้งห้าพันบาทนี่มันครอบคลุมโรคอะไรบ้างครับ?"
""".strip()

conversation_context = """\
Current Stage: Discovery 
Recent Text: "ครั้งห้าพันบาทนี่มันครอบคลุมโรคอะไรบ้างครับ? ในกรณีค่ารักษาอันนี้ครอบคลุมได้หมดทุกโรคเลยครับพี่สมชาย ยกเว้น อุบัติเหตุเท่านั้น"
""".strip()

conversation_context = """\
Current Stage: Discovery 
Recent Text: "ครอบคลุมได้หมดทุกโรคเลยครับพี่สมชาย ยกเว้น อุบัติเหตุเท่านั้น น่าสนใจอยู่นะ ไม่แน่ใจว่าอันนี้ค่าเบี้ยประกันเป็นยังไงบ้าง?"
""".strip()

conversation_context = """\
Current Stage: Discovery 
Recent Text: "ค่าเบี้ยประกันเป็นยังไงบ้าง? ค่าเบี้ยประกันตอนตัวนี้อยู่ที่เดือนละห้าร้อยบาทครับ ไม่แน่ใจว่าพี่สมชายสนใจไหม? โอน่าสนใจ"
""".strip()
conversation_context = """\
Current Stage: Discovery 
Recent Text: "ใจว่าพี่สมชายสนใจไหม? โอน่าสนใจถ้างั้นผมต้องทำยังไงบ้าง? ถ้าพี่สมชายสนใจงั้นผมคุยรายละเอียดเรื่องการสมัครเลยไหมครับ?"
""".strip()
response = sales_coach.run([{"role": "user", "content": conversation_context}])
response
# print(f"Next Action: {coaching.next_action}")
# print(f"Suggested Lines: {coaching.suggested_lines}")
# print(f"Stage Transition: {coaching.stage_transition}")


2025-11-25 20:50:52,655 - sales_coach - run:64 - INFO - start extracting...
2025-11-25 20:50:56,734 - sales_coach - run:71 - INFO - extraction successful on attempt 1
2025-11-25 20:50:56,734 - sales_coach - run:91 - INFO - end extracting with input_tokens=0 and output_tokens=0


SalesCoaching(current_stage='Discovery', next_action='stay/discovery', suggested_lines=['ยังไม่มีแผนการเลยครับ จากที่คุณเล่ามา เรามีแนวทางเหมาะกับคุณมาก'], stage_transition=<StageTransition.STAY: 'stay'>, reason="Need to gather more information about the customer's needs and goals before proceeding")

In [29]:
print(response)

current_stage='Discovery' next_action='stay/discovery' suggested_lines=['ยังไม่มีแผนการเลยครับ จากที่คุณเล่ามา เรามีแนวทางเหมาะกับคุณมาก'] stage_transition=<StageTransition.STAY: 'stay'> reason="Need to gather more information about the customer's needs and goals before proceeding"


In [19]:
conversation_context = """\
Current Stage: Discovery 
Recent Text: "ไม่แน่ใจว่าตอนนี้พี่สมชายสนใจประกันสุขภาพหรือประกันโรคร้ายแรงอยู่ไหมครับ? อืมตอนนี้ผมสนใจแต่ประกันสุขภาพครับ"
""".strip()
print(conversation_context)

Current Stage: Discovery 
Recent Text: "ไม่แน่ใจว่าตอนนี้พี่สมชายสนใจประกันสุขภาพหรือประกันโรคร้ายแรงอยู่ไหมครับ? อืมตอนนี้ผมสนใจแต่ประกันสุขภาพครับ"


In [41]:
sales_coach.calculate_cost()

{'input_tokens': 627,
 'output_tokens': 178,
 'input_cost': 0.000724185,
 'output_cost': 0.00082236,
 'total_cost': 0.0015465449999999999}

In [20]:
response = llama_llm.run(PromptHub().sales_coaching, [{"role": "user", "content": conversation_context}])
print(response.content)

**Context**: Customer has expressed interest in insurance, but there's a slight hesitation.

**COACHING RULES**:

Since the customer has shared clear needs/goals (romantic relationship), it's likely that they are ready to proceed with benefits. However, to ensure a smooth discussion and increase the chances of closing the sale, we need to address their current interest in romantic relationships.

**Next Action**: Ask about their current status regarding romantic relationships and express understanding for any challenges they might be facing.

**Specific Action**: "ครับ/ค่ะ วันนี้พี่สมชายสนใจประกันสุขภาพหรือไม่? มีอะไรอยู่ระหว่างการรักและงานบ้านครั้งใดครับ?"

**Suggested Lines**: "ตอนนี้เรามีอีกอย่างหนึ่งที่ต้องกังวลในด้านการรักและงานบ้านของพี่สมชายแล้วครับ/ค่ะ"


In [22]:
conversation_context

'Current Stage: Discovery \nRecent Text: "ไม่แน่ใจว่าตอนนี้พี่สมชายสนใจประกันสุขภาพหรือประกันโรคร้ายแรงอยู่ไหมครับ? อืมตอนนี้ผมสนใจแต่ประกันสุขภาพครับ"'

In [23]:
response = llama_llm.run(PromptHub().sales_coaching, [{"role": "user", "content": conversation_context}])
print(response.content)

**Current Stage: Discovery**
**Recent Text:** "ไม่แน่ใจว่าตอนนี้พี่สมชายสนใจประกันสุขภาพหรือประกันโรคร้ายแรงอยู่ไหมครับ? อืมตอนนี้ผมสนใจแต่ประกันสุขภาพครับ"

### Coaching Rule: Discovery Stage

**If missing key data → suggest specific questions**

The customer is clearly interested in their father's health and the agent should ask more targeted questions to gather relevant information.

```json
{
  "current_stage": "Discovery",
  "next_action": "ask_about_health_issues",
  "suggested_lines": ["คุณพี่สมชายมีปัญหาสุขภาพอยู่หรือยังไม่แน่ใจเกินไปไหมครับ?"],
  "stage_transition": "stay",
  "reason": "Need to understand health issues before recommending insurance solutions"
}
```

### Additional Suggestion

To further assess the customer's interest and needs, ask if they have any specific health concerns or questions about their father's current situation.

```json
{
  "current_stage": "Discovery",
  "next_action": "ask_about_health_issues",
  "suggested_lines": ["คุณพี่สมชายมีปัญหาสุขภาพอยู

In [19]:
response = openai_llm.run(PromptHub().sales_coaching, [{"role": "user", "content": conversation_context}])
print(response.content)

```json
{
  "current_stage": "Discovery",
  "next_action": "ask_about_budget",
  "suggested_lines": [
    "ขอให้บอกหน่อยว่า งบประมาณที่คุณสะดวกต่อการซื้อประกันต่อเดือนประมาณเท่าไหร่ครับ?",
    "ถ้าตั้งใจจะมีการชำระค่างวดประกันอย่างสม่ำเสมอ คุณคาดว่าจะใช้จำนวนเงินเท่าไหร่ต่อเดือน?"
  ],
  "stage_transition": "stay",
  "reason": "ยังขาดข้อมูลสำคัญเกี่ยวกับงบประมาณการซื้อประกัน เพื่อให้สามารถนำเสนอแผนที่เหมาะสมกับลูกค้าได้"
}
```


In [32]:
response = nova_llm.run(PromptHub().sales_coaching, [{"role": "user", "content": conversation_context}])
print(response.content)

```json
{
  "current_stage": "Discovery",
  "next_action": "ask_about_insurance_needs",
  "suggested_lines": ["จากที่ทราบแล้วว่าคุณมีภรรยาและลูกสองคน คุณมีความต้องการเกี่ยวกับประกันชีวิตหรือประกันสุขภาพใด ๆ อย่างเฉพาะอย่างไหมครับ?"],
  "stage_transition": "stay",
  "reason": "Need to understand specific insurance needs to tailor the pitch effectively"
}
```


In [33]:
from backend.llms.utils import parse_blockcode
import json
parsed_resposne = parse_blockcode(response.content, "json")
parsed_response = json.loads(parsed_resposne)

In [34]:
print(parsed_resposne)

{
  "current_stage": "Discovery",
  "next_action": "ask_about_insurance_needs",
  "suggested_lines": ["จากที่ทราบแล้วว่าคุณมีภรรยาและลูกสองคน คุณมีความต้องการเกี่ยวกับประกันชีวิตหรือประกันสุขภาพใด ๆ อย่างเฉพาะอย่างไหมครับ?"],
  "stage_transition": "stay",
  "reason": "Need to understand specific insurance needs to tailor the pitch effectively"
}


In [ ]:
SalesCoaching(**parsed_response)[]

SalesCoaching(current_stage='Discovery', next_action='ask_about_insurance_needs', suggested_lines=['จากที่ทราบแล้วว่าคุณมีภรรยาและลูกสองคน คุณมีความต้องการเกี่ยวกับประกันชีวิตหรือประกันสุขภาพใด ๆ อย่างเฉพาะอย่างไหมครับ?'], stage_transition=<StageTransition.STAY: 'stay'>, reason='Need to understand specific insurance needs to tailor the pitch effectively')

In [18]:
from backend.agents.extractor import Extractor
from backend.prompt_hub import PromptHub
from backend.llms.bedrock import BedrockNova
from backend.agents.ai_sales_coaching.extract_data_model import AgentCheckList

nova_llm = BedrockNova(model_id="us.amazon.nova-micro-v1:0")

agent_checklist_extractor = Extractor(
    agent_name="agent_checklist_extractor",
    llm=nova_llm,
    system_prompt=PromptHub().extract_agent_checklist,
    DataModel=AgentCheckList,
)

In [22]:
# content = """\
# สวัสดีครับผมอนันต์จากบริษัทประกันครับขออนุญาติเรียนสายคุณสมชายเพื่อแนะนำแบบประกันครับ
# """.strip()
# content = """\
# สวัสดีครับผมอนันต์เรียนสายคุณสมชายครับ
# """.strip()
content = """\
เรียนสายคุณสมชาย พอจะสะดวกคุยซักห้านาทีไหมครับ? สะดวกครับ
""".strip()

response = agent_checklist_extractor.run([dict(role="user", content=content)])
response

2025-11-24 21:57:25,379 - agent_checklist_extractor - run:64 - INFO - start extracting...
2025-11-24 21:57:26,632 - agent_checklist_extractor - run:71 - INFO - extraction successful on attempt 1
2025-11-24 21:57:26,633 - agent_checklist_extractor - run:91 - INFO - end extracting with input_tokens=327 and output_tokens=22


AgentCheckList(agent_introduced=None, company_mentioned=None, permission_asked=True)

In [1]:
from backend.llms.bedrock import BedrockNova
from backend.prompt_hub import PromptHub
from backend.agents.extractor import Extractor
from backend.agents.ai_sales_coaching.extract_data_model import Guide

nova_llm = BedrockNova(model_id="us.amazon.nova-micro-v1:0")

In [9]:
from backend.agents.ai_sales_coaching.extract_data_model import Guide

# 1. GREETING AGENT
greeting_agent = Extractor(
    agent_name="greeting_agent",
    llm=nova_llm,
    system_prompt=PromptHub().greeting_agent,
    DataModel=Guide,
    format="json",
)

content = """\
TEXT: สวัสดีครับผมอนันต์จากบริษัทประกันภัยครับ เรียนสายคุณสมชายครับ ไม่ทราบว่าสะดวกคุยไหมครับ? อ๋อ รีบอยู่นะครับ มีแค่ 2-3 นาทีเท่านั้น
""".strip()

response = greeting_agent.run([dict(role="user", content=content)])
print("GREETING AGENT:")
print(response.model_dump_json(indent=4))
greeting_agent.calculate_cost()

GREETING AGENT:
{
    "action": "Ask for brief permission and set time expectation",
    "explanation": "The customer expressed time pressure and gave permission to speak but also indicated a brief time frame. The agent should confirm the time constraint and assure the customer that the conversation will be brief.",
    "signals": [
        "Customer expressed time pressure",
        "Permission granted"
    ],
    "lines_to_say": [
        "ขอบคุณที่รับสาย คุณสมชาย เพราะเวลาของคุณนั้นสำคัญ ฉันจะเริ่มทันทีด้วยเวลาที่เหลือของคุณ 2-3 นาทีนี้ อยากบอกว่าในเรื่องประกันภัยมีสิ่งใหม่ๆ ที่อาจช่วยให้คุณรู้ได้"
    ]
}


{'input_tokens': 579,
 'output_tokens': 229,
 'input_cost': 0.000668745,
 'output_cost': 0.00105798,
 'total_cost': 0.0017267250000000001}

In [10]:
# 2. DISCOVERY AGENT
discovery_agent = Extractor(
    agent_name="discovery_agent",
    llm=nova_llm,
    system_prompt=PromptHub().discovery_agent,
    DataModel=Guide,
    format="json",
)

content = """\
TEXT: ไม่ทราบว่าคุณสมชายมีครอบครัวไหมครับ? มีครับ มีภรรยาแล้วก็ลูกสองคนครับ ลูกชายกับลูกสาว อายุ 8 กับ 5 ขวบครับ แล้วตอนนี้คุณมีประกันอะไรอยู่บ้างไหมครับ? มีประกันสังคมจากที่ทำงานครับ แต่รู้สึกว่ามันไม่เพียงพอน่ะครับ ถ้าเกิดอะไรขึ้นกับผม ครอบครัวจะอยู่ยังไงครับ
""".strip()

response = discovery_agent.run([dict(role="user", content=content)])
print(response.model_dump_json(indent=4))
discovery_agent.calculate_cost()

{
    "action": "Explore family protection needs",
    "explanation": "The customer has mentioned having a family, which indicates a need for family protection. It's important to delve deeper into their specific concerns and how they perceive the adequacy of their current coverage.",
    "signals": [
        "Customer revealed family situation",
        "Customer expressed concern about current coverage"
    ],
    "lines_to_say": [
        "นั่นแหละ คุณมีภรรยาและลูกสองคน การคุ้มครองครอบครัวของคุณเป็นสิ่งที่สำคัญมาก คุณมีความกังวลอย่างไรเกี่ยวกับประกันสังคมที่คุณมีอยู่ในปัจจุบัน?",
        "คุณมีความกังวลว่าหากเกิดเหตุการณ์ฉุกเฉินอะไรขึ้นกับคุณ ครอบครัวของคุณจะอยู่ได้อย่างไรหรือไม่?"
    ]
}


{'input_tokens': 642,
 'output_tokens': 281,
 'input_cost': 0.00074151,
 'output_cost': 0.0012982199999999999,
 'total_cost': 0.00203973}

In [31]:
# 3. PITCH AGENT  
pitch_agent = Extractor(
    agent_name="pitch_agent",
    llm=llama_llm,
    system_prompt=PromptHub().pitch_agent,
    DataModel=Guide,
    format="json",
)

content = """\
TEXT: ตามที่คุณสมชายบอกว่าห่วงเรื่องครอบครัว ผมเลยอยากเสนอแบบประกันชีวิตที่จะช่วยดูแลครอบครัวคุณได้ครับ มีความคุ้มครองสูงสุด 5 ล้านบาท เบี้ยประกันเดือนละ 3,500 บาท ไม่เข้าใจครับ ความคุ้มครองหมายถึงอะไร?
""".strip()

response = pitch_agent.run([dict(role="user", content=content)])
print("\nPITCH AGENT:")
print(response.model_dump_json(indent=4))
pitch_agent.calculate_cost()

NameError: name 'Guide' is not defined

In [12]:
# 4. CLOSING AGENT
closing_agent = Extractor(
    agent_name="closing_agent", 
    llm=nova_llm,
    system_prompt=PromptHub().closing_agent,
    DataModel=Guide,
    format="json",
)

content = """\
TEXT: ถ้าอย่างนั้นคุณสมชายพร้อมที่จะสมัครแบบประกันนี้ไหมครับ? ผมสนใจครับ แต่ขอคุยกับภรรยาก่อนได้ไหม? เพราะเป็นเรื่องใหญ่สำหรับครอบครัว
""".strip()

response = closing_agent.run([dict(role="user", content=content)])
print("\nCLOSING AGENT:")
print(response.model_dump_json(indent=4))
closing_agent.calculate_cost()


CLOSING AGENT:
{
    "action": "Offer to include family in discussion",
    "explanation": "The customer expresses interest but requests to discuss with his spouse first. It's important to respect this and offer a way to include the family in the decision-making process.",
    "signals": [
        "Customer needs family consultation",
        "Customer expresses interest but has a request"
    ],
    "lines_to_say": [
        "แน่นอน คุณสามารถคุยกับภรรยาได้ เราสามารถจัดเวลาพบเจอกับคุณและภรรยาเพื่อให้คุณสามารถแบ่งปันข้อมูลและตัดสินใจร่วมกันได้",
        "คุณสามารถแจ้งให้ภรรยาของคุณทราบว่าเราพร้อมรับสมัครพร้อมกันได้"
    ]
}


{'input_tokens': 538,
 'output_tokens': 241,
 'input_cost': 0.00062139,
 'output_cost': 0.00111342,
 'total_cost': 0.0017348099999999998}

In [13]:
# 5. OBJECTION HANDLING AGENT
objection_handling_agent = Extractor(
    agent_name="objection_handling_agent",
    llm=nova_llm, 
    system_prompt=PromptHub().objection_handling_agent,
    DataModel=Guide,
    format="json",
)

content = """\
TEXT: เบี้ยประกันเดือนละ 3,500 บาท แพงไปไหมครับ? เงินเดือนผมแค่ 25,000 บาท ต้องจ่ายค่าใช้จ่ายในบ้านอีกเยอะ ไม่รู้จะเหลือพอไหม
""".strip()

response = objection_handling_agent.run([dict(role="user", content=content)])
print("\nOBJECTION HANDLING AGENT:")
print(response.model_dump_json(indent=4))
objection_handling_agent.calculate_cost()


OBJECTION HANDLING AGENT:
{
    "action": "Acknowledge concern and reframe value",
    "explanation": "The customer is expressing concern about the price being too high relative to their budget. It's important to validate their concern and then reframe the value of the insurance to show how it provides long-term benefits.",
    "signals": [
        "Price objection raised",
        "Customer showing concern about affordability"
    ],
    "lines_to_say": [
        "ผมเข้าใจว่าคุณกังวลเรื่องค่าเบี้ยที่เกินรายได้ของคุณ แต่เรามีแผนที่คุ้มครองคุณในระยะยาวอย่างมีประโยชน์",
        "คุณมีคุณสมบัติที่ดีที่จะได้รับความคุ้มครองในสถานการณ์ที่สำคัญ และเราสามารถช่วยให้คุณคุมค่าได้อย่างมีประสิทธิภาพ"
    ]
}


{'input_tokens': 533,
 'output_tokens': 269,
 'input_cost': 0.000615615,
 'output_cost': 0.00124278,
 'total_cost': 0.001858395}